<a href="https://colab.research.google.com/github/Prabin-Hasham/shadenav/blob/main/notebooks/01_pipeline_hasham.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%cd /content
!rm -rf shadenav
!git clone https://github.com/Prabin-Hasham/shadenav.git
%cd shadenav

/content
Cloning into 'shadenav'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 32 (delta 7), reused 25 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 1.59 MiB | 7.76 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/shadenav


In [6]:
!pip install -q pvlib geopandas osmnx networkx

import osmnx as ox, geopandas as gpd
from datetime import datetime
from shadenav.shadows import shadow_at

G = ox.load_graphml("data/streets.graphml")
buildings = gpd.read_file("data/buildings.gpkg")

# make sure the graph is projected to match buildings (metres)
print(G.graph["crs"], "|", buildings.crs)

EPSG:32610 | EPSG:32610


In [7]:
import numpy as np
from shapely.geometry import Point

def sample_points(G, spacing_m=10):
    """For each street segment, make points every spacing_m metres along it."""
    edges = ox.graph_to_gdfs(G, nodes=False)
    rows = []
    for (u, v, k), row in edges.iterrows():
        line = row.geometry
        n = max(int(line.length // spacing_m), 1)
        for i in range(n + 1):
            pt = line.interpolate(i / n, normalized=True)
            rows.append({"u": u, "v": v, "k": k, "geometry": pt})
    return gpd.GeoDataFrame(rows, crs=edges.crs)

samples = sample_points(G)
print(len(samples), "sample points along", G.number_of_edges(), "segments")

15066 sample points along 4154 segments


In [8]:
def score_edges(G, samples, shadow_geom):
    """Fraction of each segment's sample points that fall in shade -> 'sun_fraction' per edge."""
    if shadow_geom is None:
        # sun is down: treat everything as fully shaded (no sun to avoid)
        for u, v, k in G.edges(keys=True):
            G[u][v][k]["sun_fraction"] = 0.0
        return G

    # which sample points are inside the shadow?
    inside = samples.within(shadow_geom)
    samples = samples.assign(shaded=inside.values)

    # average the shaded flag per segment
    frac_shaded = samples.groupby(["u", "v", "k"])["shaded"].mean()

    for (u, v, k), shaded in frac_shaded.items():
        G[u][v][k]["sun_fraction"] = 1.0 - shaded  # sun_fraction = 1 - shade
    return G

In [9]:
shadow = shadow_at(buildings, datetime(2025, 9, 15, 9))
G = score_edges(G, samples, shadow)

# look at the spread
import numpy as np
fracs = [d["sun_fraction"] for _, _, d in G.edges(data=True) if "sun_fraction" in d]
print(f"{len(fracs)} edges scored")
print(f"sunniest: {np.max(fracs):.2f}, shadiest: {np.min(fracs):.2f}, mean: {np.mean(fracs):.2f}")

4154 edges scored
sunniest: 1.00, shadiest: 0.00, mean: 0.80


In [10]:
import networkx as nx

def shade_route(G, orig_pt, dest_pt, lam=1.0):
    """Route from orig to dest, penalizing sun exposure by lam.

    cost = length + lam * length * sun_fraction
      lam = 0  -> plain shortest path
      lam high -> detours to stay in shade
    """
    # nearest graph nodes to the clicked points
    orig = ox.distance.nearest_nodes(G, orig_pt[0], orig_pt[1])
    dest = ox.distance.nearest_nodes(G, dest_pt[0], dest_pt[1])

    # build the blended weight on every edge
    for u, v, k, d in G.edges(keys=True, data=True):
        sun = d.get("sun_fraction", 1.0)
        d["cost"] = d["length"] * (1 + lam * sun)

    path = nx.shortest_path(G, orig, dest, weight="cost")

    # measure what we got
    total_len = sum(
        min(G[u][v][kk]["length"] for kk in G[u][v])
        for u, v in zip(path[:-1], path[1:])
    )
    total_sun = sum(
        min((G[u][v][kk]["length"] * G[u][v][kk].get("sun_fraction", 1.0))
            for kk in G[u][v])
        for u, v in zip(path[:-1], path[1:])
    )
    return path, total_len, total_sun

In [11]:
nodes = ox.graph_to_gdfs(G, edges=False)
pts = nodes.geometry
orig_pt = (pts.iloc[0].x, pts.iloc[0].y)
dest_pt = (pts.iloc[-1].x, pts.iloc[-1].y)

for lam in [0, 1, 3]:
    path, length, sun = shade_route(G, orig_pt, dest_pt, lam=lam)
    print(f"lam={lam}: {length:.0f}m total, {sun:.0f}m in sun "
          f"({100*sun/length:.0f}% sunny)")

lam=0: 708m total, 647m in sun (91% sunny)
lam=1: 712m total, 518m in sun (73% sunny)
lam=3: 712m total, 518m in sun (73% sunny)


In [12]:
nodes = ox.graph_to_gdfs(G, edges=False)
xs, ys = nodes.geometry.x, nodes.geometry.y

# far apart: SW corner to NE corner
orig_pt = (xs.min(), ys.min())
dest_pt = (xs.max(), ys.max())

for lam in [0, 2, 5, 10]:
    path, length, sun = shade_route(G, orig_pt, dest_pt, lam=lam)
    print(f"lam={lam:2d}: {length:.0f}m total, {sun:.0f}m in sun ({100*sun/length:.0f}% sunny)")

lam= 0: 1722m total, 1510m in sun (88% sunny)
lam= 2: 1897m total, 1122m in sun (59% sunny)
lam= 5: 1926m total, 1116m in sun (58% sunny)
lam=10: 1926m total, 1116m in sun (58% sunny)


In [13]:
import folium

def route_latlon(G, path):
    nodes = ox.graph_to_gdfs(G, edges=False).to_crs(4326)
    return [(nodes.loc[n].geometry.y, nodes.loc[n].geometry.x) for n in path]

path_short, len_s, sun_s = shade_route(G, orig_pt, dest_pt, lam=0)
path_shade, len_h, sun_h = shade_route(G, orig_pt, dest_pt, lam=2)

ctr = ox.graph_to_gdfs(G, edges=False).to_crs(4326).geometry
m = folium.Map(location=[ctr.y.mean(), ctr.x.mean()], zoom_start=15)

folium.PolyLine(route_latlon(G, path_short), color="red", weight=5, opacity=0.7,
                tooltip=f"Shortest: {len_s:.0f}m, {100*sun_s/len_s:.0f}% sun").add_to(m)
folium.PolyLine(route_latlon(G, path_shade), color="green", weight=5, opacity=0.7,
                tooltip=f"Shady: {len_h:.0f}m, {100*sun_h/len_h:.0f}% sun").add_to(m)
folium.Marker(route_latlon(G, path_short)[0], tooltip="Start").add_to(m)
folium.Marker(route_latlon(G, path_short)[-1], tooltip="End").add_to(m)
m